# Fine-tuning MiniFASNetV2 cho bài toán Face Anti-Spoofing

Notebook thực nghiệm thật, **chưa chạy training**. Giữ notebook CNN cũ riêng biệt. Không tự ghi đè ONNX production.

## Vì sao pretrained?
Dữ liệu local ít, nhiều ảnh lặp và chủ thể real hạn chế; huấn luyện từ đầu dễ học nền/thiết bị/chủ thể thay vì dấu hiệu PAD. Pretrained là điểm khởi đầu, không bảo đảm tổng quát hóa. Fine-tuning cần dữ liệu có nhãn và provenance, split không leak; validation/test độc lập. Không tuyên bố VShield đã train model gốc.

Nguồn kiến trúc/checkpoint: [MiniVision Silent-Face-Anti-Spoofing](https://github.com/minivision-ai/Silent-Face-Anti-Spoofing/tree/b6d5f04ad78778917853b25c778acef6d5626d15), được hash-pin trong `scripts/setup-minifasnet.py`. Kiểm tra điều kiện license upstream trước phân phối.


## 1. Cấu trúc model (V2, không phải V2SE)

BGR 80×80 → Conv3×3 stride2 → depthwise3×3 → bottleneck downsample → 4 residual bottlenecks → downsample → 6 residual bottlenecks → downsample → 2 residual bottlenecks → Conv1×1 (512) → depthwise5×5 → Flatten512 → Linear128 → BN → Dropout0.2 → Linear3.

Bottleneck: pointwise1×1 + BN/PReLU → depthwise3×3 + BN/PReLU → pointwise1×1 + BN; cộng residual khi cấu hình cho phép. Model có 3 logits: spoof0 / real(index1) / spoof2. Nhãn local chỉ fake/real: gộp 2 spoof logits bằng logsumexp, không tự gán subtype fake.

Chọn V2 để giữ đúng checkpoint và inference contract đang có trong repo; kiến trúc depthwise nhỏ phù hợp ứng dụng CPU. **Không có benchmark chứng minh đây là model tốt nhất** cho dữ liệu này.


In [ ]:
from pathlib import Path
import sys, json, csv, hashlib, importlib.util, random, platform
import numpy as np
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "scripts/setup-minifasnet.py").is_file(), "Open from project/notebooks"
sys.path.insert(0, str(ROOT / "src"))
RUN_TRAINING = False
RUN_TEST = False
SEED, EPOCHS, WARMUP, BATCH_SIZE = 42, 10, 2, 16
HEAD_LR, FINETUNE_LR, THRESHOLD = 1e-3, 1e-5, 0.8
DATA_CONFIG = ROOT / "configs/data-v2.yaml"
print("Training disabled by default; hyperparameters proposed, not calibrated.")


## 2. Dependencies và pretrained weights

Chọn kernel của `.venv`. Cần PyTorch và gói `lightning` (import `lightning.pytorch`), cv2, numpy, pyyaml, matplotlib. Cài PyTorch phù hợp CPU/CUDA theo [hướng dẫn chính thức](https://pytorch.org/get-started/locally/), sau đó chạy `%pip install "lightning>=2.5,<3"` trong đúng kernel nếu cần; notebook không tự cài/download.

Fine-tuning dùng [LightningModule và Trainer](https://lightning.ai/docs/pytorch/stable/common/lightning_module.html), ModelCheckpoint chọn validation loss và CSVLogger ghi metrics. Cache upstream phải tồn tại và đúng hash; không chạy main setup script vì main ghi model production.


In [ ]:
# Restart kernel before running this cell if CUDA was already initialized.
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, Callback
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import Dataset, DataLoader
import cv2, yaml
from vshield.core.pad_preprocessing import prepare_pad_input
from vshield.data.loader import _validate_release_manifest
pl.seed_everything(SEED, workers=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
spec = importlib.util.spec_from_file_location("verified_pad_setup", ROOT / "scripts/setup-minifasnet.py")
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
model = setup.reference_model(download=False).to(DEVICE)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()), "Device:", DEVICE)
with torch.inference_mode():
    assert model(torch.zeros(2, 3, 80, 80, device=DEVICE)).shape == (2, 3)


## 3. Data và release gate

Local consent do người dùng cung cấp; không tự suy diễn subject/session/device từ tên file. `configs/data-v2.yaml` hiện pending release; notebook chặn training cho đến khi release hợp lệ. Dữ liệu Axon tải về vẫn là candidate/quarantine, không tự thêm vào train; license CC BY-NC4.0 và nhãn bona-fide/provenance phải xác minh.

Protocol phải có `status=released`, split, relative_path, sha256, label(0=fake,1=real), subject_id/session_id/clip_id/device_id; thêm **bbox_x,bbox_y,bbox_w,bbox_h** được đo/xác minh trên ảnh gốc. Sau khi bổ sung bbox phải release và đăng ký hash protocol lại, không sửa registry bằng tay để bỏ gate. Không dùng cả ảnh làm bbox giả. Near-duplicate/shortcut audit nằm trong quy trình release của repo.

Preprocessing dùng chính production function: crop context2.7, BGR, 80×80, float32 NCHW, giá trị0–255 (**không /255**). Không dùng loader CNN128×128 cũ.


In [ ]:
def load_protocols(config_path):
    cfg = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    if cfg.get("status") != "released":
        raise ValueError("Dataset not released; resolve provenance/leakage before training.")
    _validate_release_manifest(config_path.parent, cfg)
    root = (config_path.parent / cfg["dataset_root"]).resolve()
    splits, seen = {}, {key: {} for key in ("sha256", "subject_id", "session_id", "clip_id", "device_id")}
    for split in ("train", "val", "test"):
        path = (config_path.parent / cfg["protocols"][split]).resolve()
        with path.open(encoding="utf-8-sig", newline="") as stream:
            rows = list(csv.DictReader(stream))
        if not rows:
            raise ValueError(f"Empty {split}")
        for row in rows:
            if row["status"] != "released" or row["split"] != split or row["label"] not in ("0", "1"):
                raise ValueError("Invalid released label/split")
            image_path = (root / row["relative_path"]).resolve()
            if not image_path.is_relative_to(root):
                raise ValueError("Image path escapes dataset root")
            if hashlib.sha256(image_path.read_bytes()).hexdigest() != row["sha256"]:
                raise ValueError("Image hash mismatch")
            for key, groups in seen.items():
                value = row.get(key, "").strip()
                if not value or value.lower() in ("unknown", "todo"):
                    raise ValueError(f"Missing provenance: {key}")
                if value in groups and groups[value] != split:
                    raise ValueError(f"Cross-split leakage: {key}")
                groups[value] = split
            box = [int(row[key]) for key in ("bbox_x", "bbox_y", "bbox_w", "bbox_h")]
            prepare_pad_input(cv2.imread(str(image_path)), box)
            row["_path"], row["_bbox"] = str(image_path), box
        if {r["label"] for r in rows} != {"0", "1"}:
            raise ValueError(f"Both classes required in {split}")
        splits[split] = rows
    return cfg, splits

data_error = None
try:
    data_config, split_rows = load_protocols(DATA_CONFIG)
    print({k: len(v) for k, v in split_rows.items()})
except (ValueError, KeyError, OSError) as exc:
    data_error = str(exc)
    print("DATA BLOCKED:", data_error)
    if RUN_TRAINING or RUN_TEST:
        raise


In [ ]:
class PADFrames(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        tensor = prepare_pad_input(cv2.imread(row["_path"]), row["_bbox"])[0]
        return torch.from_numpy(tensor), torch.tensor(float(row["label"]))

def binary_logit(logits):
    return logits[:, 1] - torch.logsumexp(logits[:, [0, 2]], dim=1)

criterion = nn.BCEWithLogitsLoss()
# Algebra contract: sigmoid(binary_logit) equals original softmax(real).
probe = torch.tensor([[0., 2., -1.], [5., 0., 3.]])
torch.testing.assert_close(torch.sigmoid(binary_logit(probe)), torch.softmax(probe, 1)[:, 1])

def evaluate(network, loader):
    network.eval()
    loss_sum, count, tp, tn, fp, fn = 0., 0, 0, 0, 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            scores = binary_logit(network(images))
            loss_sum += criterion(scores, labels).item() * len(labels)
            pred, real = torch.sigmoid(scores) >= THRESHOLD, labels.bool()
            tp += int((pred & real).sum()); tn += int((~pred & ~real).sum())
            fp += int((pred & ~real).sum()); fn += int((~pred & real).sum())
            count += len(labels)
    return dict(loss=loss_sum/count, n=count, tp=tp, tn=tn, fp=fp, fn=fn,
                accuracy=(tp+tn)/count, spoof_accepted=fp/(fp+tn), real_rejected=fn/(fn+tp))


## 4. Fine-tune bằng PyTorch Lightning

`MiniFASNetFinetuner(LightningModule)` định nghĩa training_step, validation_step và AdamW; `Trainer.fit` quản lý device, backward, optimizer và gradient clipping. Warm-up classifier rồi unfreeze backbone với LR thấp. BN statistics luôn freeze, Dropout vẫn train. Batch>=2 và drop_last; ghi số ảnh thực sự xử lý.

`ModelCheckpoint(monitor="val_loss", mode="min")` chọn checkpoint; `CSVLogger` và callback lưu lịch sử thật. Sau fit, xuất weights tốt nhất vào `best-candidate.pth` để tương thích cell đánh giá. Threshold0.8 chưa calibrated; test chỉ bật riêng sau khi chốt cấu hình. Không tự thay model production.


In [ ]:
class MiniFASNetFinetuner(pl.LightningModule):
    def __init__(self, network):
        super().__init__()
        self.network = network
        self.train_total = 0.
        self.train_count = 0
        for name, parameter in self.network.named_parameters():
            parameter.requires_grad = name.startswith("prob.")

    def forward(self, images):
        return self.network(images)

    def configure_optimizers(self):
        # Include frozen params now so unfreezing does not require a new optimizer.
        head, backbone = [], []
        for name, parameter in self.network.named_parameters():
            (head if name.startswith("prob.") else backbone).append(parameter)
        return torch.optim.AdamW([
            {"params": head, "lr": HEAD_LR},
            {"params": backbone, "lr": FINETUNE_LR},
        ], weight_decay=1e-4)

    def on_train_epoch_start(self):
        self.train_total, self.train_count = 0., 0
        if self.current_epoch >= WARMUP:
            for parameter in self.network.parameters():
                parameter.requires_grad = True
            for group in self.trainer.optimizers[0].param_groups:
                group["lr"] = FINETUNE_LR

    def on_train_batch_start(self, batch, batch_idx):
        # Trainer restores train mode; freeze running statistics again each batch.
        for module in self.network.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()

    def training_step(self, batch, batch_idx):
        images, labels = batch
        loss = criterion(binary_logit(self(images)), labels)
        if not torch.isfinite(loss):
            raise ValueError("Non-finite training loss")
        self.train_total += float(loss.detach()) * len(labels)
        self.train_count += len(labels)
        self.log("train_loss", loss, on_step=False, on_epoch=True, batch_size=len(labels))
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        loss = criterion(binary_logit(self(images)), labels)
        if not torch.isfinite(loss):
            raise ValueError("Non-finite validation loss")
        self.log("val_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=len(labels))

class RecordHistory(Callback):
    def on_validation_end(self, trainer, module):
        if trainer.sanity_checking:
            return
        row = dict(epoch=trainer.current_epoch+1,
                   train_loss=module.train_total/module.train_count,
                   processed=module.train_count,
                   val_loss=float(trainer.callback_metrics["val_loss"]))
        history.append(row)
        (run_dir / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
        with (run_dir / "history.csv").open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(row))
            writer.writeheader()
            writer.writerows(history)
        print(row)

history, run_dir = [], None
if RUN_TRAINING:
    pl.seed_everything(SEED, workers=True)
    # Each run starts from the verified original, never a mutated prior run.
    model = setup.reference_model(download=False).to(DEVICE)
    if data_error:
        raise ValueError(data_error)
    if BATCH_SIZE < 2 or len(split_rows["train"]) < BATCH_SIZE or not 0 < WARMUP < EPOCHS:
        raise ValueError("Require valid batch size and 0 < WARMUP < EPOCHS")
    from datetime import datetime, timezone
    from uuid import uuid4
    run_dir = ROOT / "outputs" / ("minifasnet-finetune-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
    run_dir.mkdir(parents=True, exist_ok=False)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(PADFrames(split_rows["train"]), batch_size=BATCH_SIZE,
                              shuffle=True, drop_last=True, num_workers=0, generator=generator)
    val_loader = DataLoader(PADFrames(split_rows["val"]), batch_size=BATCH_SIZE, num_workers=0)
    metadata = dict(seed=SEED, epochs=EPOCHS, warmup=WARMUP, batch_size=BATCH_SIZE,
                    head_lr=HEAD_LR, finetune_lr=FINETUNE_LR, threshold=THRESHOLD,
                    source_commit=setup.COMMIT, checkpoint_sha=setup.WEIGHTS_SHA,
                    architecture_sha=setup.SOURCE_SHA, torch=str(torch.__version__), lightning=str(pl.__version__),
                    python=platform.python_version(), device=str(DEVICE),
                    preprocessing="production prepare_pad_input BGR 0..255 NCHW 80x80 scale2.7",
                    config_sha=hashlib.sha256(DATA_CONFIG.read_bytes()).hexdigest(),
                    protocol_hashes={s: hashlib.sha256((DATA_CONFIG.parent / p).resolve().read_bytes()).hexdigest()
                                     for s, p in data_config["protocols"].items()})
    (run_dir / "manifest.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    baseline = evaluate(model, val_loader)
    (run_dir / "baseline-validation.json").write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    checkpoint = ModelCheckpoint(
        dirpath=str(run_dir / "checkpoints"), filename="best-{epoch:02d}",
        monitor="val_loss", mode="min", save_top_k=1, save_last=False,
        save_on_train_epoch_end=False)
    csv_logger = CSVLogger(save_dir=str(run_dir), name="lightning", version=0)
    trainer = pl.Trainer(
        max_epochs=EPOCHS, accelerator="gpu" if DEVICE.type == "cuda" else "cpu",
        devices=1, precision="32-true", deterministic=True,
        gradient_clip_val=1., callbacks=[checkpoint, RecordHistory()],
        logger=csv_logger, default_root_dir=str(run_dir),
        num_sanity_val_steps=0, log_every_n_steps=1)
    finetuner = MiniFASNetFinetuner(model)
    trainer.fit(finetuner, train_dataloaders=train_loader, val_dataloaders=val_loader)
    if not checkpoint.best_model_path:
        raise RuntimeError("No validation-selected checkpoint")
    best_state = torch.load(checkpoint.best_model_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(
        {key.removeprefix("network."): value for key, value in best_state["state_dict"].items()},
        strict=True)
    model.to(DEVICE)
    torch.save(dict(state_dict=model.state_dict(), epoch=int(best_state["epoch"])+1,
                    validation=evaluate(model, val_loader), metadata=metadata),
               run_dir / "best-candidate.pth")
    print("Best Lightning checkpoint:", checkpoint.best_model_path)
else:
    print("No training executed. Enable RUN_TRAINING only after dataset release.")


In [ ]:
if history:
    import matplotlib.pyplot as plt
    plt.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="train")
    plt.plot([r["epoch"] for r in history], [r["val_loss"] for r in history], label="validation")
    plt.xlabel("Epoch"); plt.ylabel("Binary PAD loss"); plt.legend()
    plt.savefig(run_dir / "loss.png", bbox_inches="tight")
    plt.show()
else:
    print("No history to plot; no invented learning curve.")


## 5. Test cuối cùng — không dùng để chọn hyperparameter

Chốt checkpoint/threshold trước khi bật RUN_TEST. Confusion counts và tỷ lệ dưới đây chỉ đại diện dataset released này, không chứng minh độ chính xác toàn hệ thống. Không tune trên test. Đưa candidate vào production cần ONNX export/parity, calibration, regression và approval riêng.


In [ ]:
# For a new kernel, set CANDIDATE_RUN to the actual completed run directory.
CANDIDATE_RUN = run_dir
if RUN_TEST:
    if data_error:
        raise ValueError(data_error)
    if CANDIDATE_RUN is None:
        raise ValueError("Set actual candidate run directory")
    candidate_path = Path(CANDIDATE_RUN).resolve()
    if not candidate_path.is_relative_to((ROOT / "outputs").resolve()):
        raise ValueError("Candidate must be in outputs")
    result_path = candidate_path / "final-test.json"
    if result_path.exists():
        raise ValueError("Test already recorded; do not overwrite/tune on test")
    state = torch.load(candidate_path / "best-candidate.pth", map_location=DEVICE, weights_only=True)
    if state["metadata"]["threshold"] != THRESHOLD:
        raise ValueError("Threshold changed after training")
    for split, protocol in data_config["protocols"].items():
        digest = hashlib.sha256((DATA_CONFIG.parent / protocol).resolve().read_bytes()).hexdigest()
        if digest != state["metadata"]["protocol_hashes"][split]:
            raise ValueError("Dataset protocol changed after training")
    test_loader = DataLoader(PADFrames(split_rows["test"]), batch_size=BATCH_SIZE, num_workers=0)
    baseline_model = setup.reference_model(download=False).to(DEVICE)
    baseline_test = evaluate(baseline_model, test_loader)
    model.load_state_dict(state["state_dict"], strict=True)
    results = dict(baseline=baseline_test, candidate=evaluate(model, test_loader), threshold=THRESHOLD)
    result_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
    print(results)
else:
    print("Final test disabled.")
